# Traces analysis

Basic pandas / numpy look at `traces-1787841590147.json` (Jaeger / OpenTelemetry export).

In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

DATA_PATH = Path("traces-1787841590147.json")

with DATA_PATH.open(encoding="utf-8") as f:
    payload = json.load(f)

traces = payload["data"]
print(f"Loaded {len(traces)} traces from {DATA_PATH.name}")

Loaded 8 traces from traces-1787841590147.json


## Flatten spans into a DataFrame

In [11]:
def tags_to_dict(tags):
    return {t["key"]: t["value"] for t in tags or []}


rows = []
for trace in traces:
    processes = trace.get("processes") or {}
    for span in trace.get("spans") or []:
        tags = tags_to_dict(span.get("tags"))
        process = span.get("process") or processes.get(span.get("processID"), {})
        rows.append(
            {
                "trace_id": span.get("traceID"),
                "span_id": span.get("spanID"),
                "operation": span.get("operationName"),
                "service": process.get("serviceName"),
                "start_time_us": span.get("startTime"),
                "duration_us": span.get("duration"),
                "depth": span.get("depth"),
                "has_children": span.get("hasChildren"),
                "span_kind": tags.get("span.kind"),
                "event_type": tags.get("analytics.event_type"),
                "enduser_id": tags.get("enduser.id"),
                "http_method": tags.get("http.method"),
                "http_status": tags.get("http.status_code"),
                "http_url": tags.get("http.url"),
                "otel_scope": tags.get("otel.scope.name"),
                "error": tags.get("error"),
            }
        )

spans = pd.DataFrame(rows)
spans["duration_ms"] = spans["duration_us"] / 1000.0
spans["start_time"] = pd.to_datetime(spans["start_time_us"], unit="us", utc=True)

spans.head()

,trace_id,span_id,operation,service,start_time_us,duration_us,depth,has_children,span_kind,event_type,enduser_id,http_method,http_status,http_url,otel_scope,error,duration_ms,start_time
0,b6deb6117b3164e4c82057d57870b0b8,70c0da4ac74956c7,analytics.store_event,brevity-analytics,1787841569626286,4719,2,False,internal,like_created,NaN,NaN,NaN,NaN,brevity-analytics,None,4.719,2026-08-27 14:39:29.626286+00:00
1,b6deb6117b3164e4c82057d57870b0b8,50023c8e5c6aa937,POST,brevity-api,1787841569622300,10474,1,True,client,NaN,NaN,POST,201.0,http://brevity-analytics:8001/analytics/event,opentelemetry.instrumentation.httpx,None,10.474,2026-08-27 14:39:29.622300+00:00
2,b6deb6117b3164e4c82057d57870b0b8,7a5e04444a4c844b,analytics.emit,brevity-api,1787841569567798,66130,0,True,internal,like_created,1413cfaf-0290-4bb3-b517-a318ca17ea34,NaN,NaN,NaN,brevity-api.analytics,None,66.130,2026-08-27 14:39:29.567798+00:00
3,8a78e8416426bbd551c73bea1017ed7c,2497c2ec3193c115,POST,brevity-api,1787841570748638,9814,1,True,client,NaN,NaN,POST,201.0,http://brevity-analytics:8001/analytics/event,opentelemetry.instrumentation.httpx,None,9.814,2026-08-27 14:39:30.748638+00:00
4,8a78e8416426bbd551c73bea1017ed7c,4c1bcc73cd39a5f0,analytics.emit,brevity-api,1787841570738295,21035,0,True,internal,like_removed,1413cfaf-0290-4bb3-b517-a318ca17ea34,NaN,NaN,NaN,brevity-api.analytics,None,21.035,2026-08-27 14:39:30.738295+00:00


## Dataset overview

In [12]:
print("shape:", spans.shape)
print("columns:", list(spans.columns))
print()
print("traces:", spans["trace_id"].nunique())
print("spans:", len(spans))
print("services:", spans["service"].nunique())
print("operations:", spans["operation"].nunique())
print()
display(spans[["trace_id", "service", "operation", "duration_ms", "event_type", "http_status"]].head(20))

shape: (16, 18)
columns: ['trace_id', 'span_id', 'operation', 'service', 'start_time_us', 'duration_us', 'depth', 'has_children', 'span_kind', 'event_type', 'enduser_id', 'http_method', 'http_status', 'http_url', 'otel_scope', 'error', 'duration_ms', 'start_time']

traces: 8
spans: 16
services: 2
operations: 4



,trace_id,service,operation,duration_ms,event_type,http_status
0,b6deb6117b3164e4c82057d57870b0b8,brevity-analytics,analytics.store_event,4.719,like_created,NaN
1,b6deb6117b3164e4c82057d57870b0b8,brevity-api,POST,10.474,NaN,201.0
2,b6deb6117b3164e4c82057d57870b0b8,brevity-api,analytics.emit,66.130,like_created,NaN
3,8a78e8416426bbd551c73bea1017ed7c,brevity-api,POST,9.814,NaN,201.0
4,8a78e8416426bbd551c73bea1017ed7c,brevity-api,analytics.emit,21.035,like_removed,NaN
5,8a78e8416426bbd551c73bea1017ed7c,brevity-analytics,analytics.store_event,5.297,like_removed,NaN
6,5b7ed325b0b82407c0abe197360e6024,brevity-api,BackgroundTask emit_event,21.624,NaN,NaN
7,aeaa8425413340e630537cdc458d4256,brevity-analytics,analytics.store_event,5.049,like_removed,NaN
8,aeaa8425413340e630537cdc458d4256,brevity-api,POST,10.651,NaN,201.0
9,aeaa8425413340e630537cdc458d4256,brevity-api,analytics.emit,20.504,like_removed,NaN


## Duration stats (numpy)

In [13]:
durations_ms = spans["duration_ms"].to_numpy(dtype=float)

stats = {
    "count": durations_ms.size,
    "mean_ms": float(np.mean(durations_ms)),
    "median_ms": float(np.median(durations_ms)),
    "std_ms": float(np.std(durations_ms)),
    "min_ms": float(np.min(durations_ms)),
    "p50_ms": float(np.percentile(durations_ms, 50)),
    "p90_ms": float(np.percentile(durations_ms, 90)),
    "p99_ms": float(np.percentile(durations_ms, 99)),
    "max_ms": float(np.max(durations_ms)),
}

pd.Series(stats).round(3)

count        16.000
mean_ms      18.963
median_ms    12.529
std_ms       18.979
min_ms        3.748
p50_ms       12.529
p90_ms       43.877
p99_ms       66.766
max_ms       66.878
dtype: float64

## Breakdowns by service and operation

In [14]:
by_service = (
    spans.groupby("service", dropna=False)["duration_ms"]
    .agg(count="count", mean_ms="mean", median_ms="median", max_ms="max")
    .round(3)
    .sort_values("count", ascending=False)
)
by_service

,count,mean_ms,median_ms,max_ms
service,,,,
brevity-api,12,23.716,17.930,66.878
brevity-analytics,4,4.703,4.884,5.297


In [15]:
by_operation = (
    spans.groupby(["service", "operation"], dropna=False)["duration_ms"]
    .agg(count="count", mean_ms="mean", median_ms="median", p90_ms=lambda s: np.percentile(s, 90))
    .round(3)
    .sort_values("mean_ms", ascending=False)
)
by_operation

count  mean_ms  median_ms  p90_ms
service           operation                                                   
brevity-api       BackgroundTask emit_event      4   31.227     21.338  53.302
                  analytics.emit                 4   30.519     20.770  52.602
                  POST                           4    9.402     10.144  10.598
brevity-analytics analytics.store_event          4    4.703      4.884   5.223

## Analytics events and HTTP

In [16]:
print("event_type counts")
display(spans["event_type"].value_counts(dropna=False))

print("\nspan_kind counts")
display(spans["span_kind"].value_counts(dropna=False))

print("\nhttp status / method")
display(
    spans.dropna(subset=["http_method"])[["operation", "http_method", "http_status", "http_url", "duration_ms"]]
)

error_count = spans["error"].notna().sum()
print(f"\nspans with error tag: {error_count}")

event_type counts


event_type
NaN             8
like_created    4
like_removed    4
Name: count, dtype: int64


span_kind counts


span_kind
internal    12
client       4
Name: count, dtype: int64


http status / method


,operation,http_method,http_status,http_url,duration_ms
1,POST,POST,201.0,http://brevity-analytics:8001/analytics/event,10.474
3,POST,POST,201.0,http://brevity-analytics:8001/analytics/event,9.814
8,POST,POST,201.0,http://brevity-analytics:8001/analytics/event,10.651
11,POST,POST,201.0,http://brevity-analytics:8001/analytics/event,6.668



spans with error tag: 0


## Trace-level summary

In [17]:
trace_summary = (
    spans.groupby("trace_id")
    .agg(
        span_count=("span_id", "count"),
        services=("service", lambda s: ", ".join(sorted(set(s.dropna())))),
        operations=("operation", lambda s: ", ".join(sorted(set(s.dropna())))),
        event_types=("event_type", lambda s: ", ".join(sorted({x for x in s.dropna()})) or None),
        total_duration_ms=("duration_ms", "sum"),
        max_span_ms=("duration_ms", "max"),
        start=("start_time", "min"),
    )
    .sort_values("start")
)

trace_summary

,span_count,services,operations,event_types,total_duration_ms,max_span_ms,start
trace_id,,,,,,,
0af75274efd951220a0c5a0e4a4a89b8,1,brevity-api,BackgroundTask emit_event,NaN,66.878,66.878,2026-08-27 14:39:29.567216+00:00
b6deb6117b3164e4c82057d57870b0b8,3,"brevity-analytics, brevity-api","POST, analytics.emit, analytics.store_event",like_created,81.323,66.130,2026-08-27 14:39:29.567798+00:00
e0fbcfba9adf7b2d22f21408e76ddc0a,1,brevity-api,BackgroundTask emit_event,NaN,21.052,21.052,2026-08-27 14:39:29.885272+00:00
aeaa8425413340e630537cdc458d4256,3,"brevity-analytics, brevity-api","POST, analytics.emit, analytics.store_event",like_removed,36.204,20.504,2026-08-27 14:39:29.885590+00:00
7f09943f67a8f4dfa3a36c4a64753073,1,brevity-api,BackgroundTask emit_event,NaN,15.355,15.355,2026-08-27 14:39:30.490639+00:00
0fbe1ca994b8f8461522a24c13623889,3,"brevity-analytics, brevity-api","POST, analytics.emit, analytics.store_event",like_created,24.824,14.408,2026-08-27 14:39:30.491393+00:00
5b7ed325b0b82407c0abe197360e6024,1,brevity-api,BackgroundTask emit_event,NaN,21.624,21.624,2026-08-27 14:39:30.737909+00:00
8a78e8416426bbd551c73bea1017ed7c,3,"brevity-analytics, brevity-api","POST, analytics.emit, analytics.store_event",like_removed,36.146,21.035,2026-08-27 14:39:30.738295+00:00


## Slowest spans

In [18]:
(
    spans.sort_values("duration_ms", ascending=False)[
        ["trace_id", "service", "operation", "event_type", "duration_ms", "start_time"]
    ]
    .head(10)
    .reset_index(drop=True)
)

,trace_id,service,operation,event_type,duration_ms,start_time
0,0af75274efd951220a0c5a0e4a4a89b8,brevity-api,BackgroundTask emit_event,NaN,66.878,2026-08-27 14:39:29.567216+00:00
1,b6deb6117b3164e4c82057d57870b0b8,brevity-api,analytics.emit,like_created,66.130,2026-08-27 14:39:29.567798+00:00
2,5b7ed325b0b82407c0abe197360e6024,brevity-api,BackgroundTask emit_event,NaN,21.624,2026-08-27 14:39:30.737909+00:00
3,e0fbcfba9adf7b2d22f21408e76ddc0a,brevity-api,BackgroundTask emit_event,NaN,21.052,2026-08-27 14:39:29.885272+00:00
4,8a78e8416426bbd551c73bea1017ed7c,brevity-api,analytics.emit,like_removed,21.035,2026-08-27 14:39:30.738295+00:00
5,aeaa8425413340e630537cdc458d4256,brevity-api,analytics.emit,like_removed,20.504,2026-08-27 14:39:29.885590+00:00
6,7f09943f67a8f4dfa3a36c4a64753073,brevity-api,BackgroundTask emit_event,NaN,15.355,2026-08-27 14:39:30.490639+00:00
7,0fbe1ca994b8f8461522a24c13623889,brevity-api,analytics.emit,like_created,14.408,2026-08-27 14:39:30.491393+00:00
8,aeaa8425413340e630537cdc458d4256,brevity-api,POST,NaN,10.651,2026-08-27 14:39:29.894123+00:00
9,b6deb6117b3164e4c82057d57870b0b8,brevity-api,POST,NaN,10.474,2026-08-27 14:39:29.622300+00:00
